# NK13 analysis II
- Load modules and libraries
- Plot 1:
- Plot 2:

In [ ]:
import anndata as ad
import scanpy as sc
import scirpy as ir
import pandas as pd
import numpy as np
import seaborn as sb
import muon as mu
# from scipy.stats import binomtest
from scipy.stats import mannwhitneyu
# import string
from matplotlib.patches import Patch


# Helps to define plotting area
import matplotlib.pyplot as plt
# from matplotlib import rcParams 
# from matplotlib.pyplot import rc_context
# import matplotlib.gridspec as gridspec
# import matplotlib.colors as mcolors
# from matplotlib import colors
# # from matplotlib_venn import venn2
# from scipy.stats import spearmanr
# import matplotlib.patches as mpatches
# from pathlib import Path

## For making text editable in .pdfs once exported use cairo backend instead of agg
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# set seed
np.random.seed(44)

#display options
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999

# Loading data

In [ ]:
mdata = mu.read(filename="../../04_data_objects/06_preprocessed_v2/NK13_PostV_scANVI_labeltransfer_intergration.h5mu")
len(mdata)

In [ ]:
#Group annotation
patient_list = [ "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1","Pt8", "Pt18", "Pt19", "Pt21"]

#Match the lists defined above to define "RRMS" and "Control"
mdata["gex"].obs["catpat"] = pd.Categorical(np.where(mdata["gex"].obs["donor"].isin(patient_list), "MS", np.where(mdata["gex"].obs["donor"].isin(control_list), "NIC", "IC")))


# Report data

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(mdata["gex"], color=["L3_ann"],   
           legend_fontoutline=2, 
           legend_fontweight='bold', 
           legend_fontsize="6", 
           color_map = 'RdYlBu_r',
           frameon=False, 
           vmin=0)

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(mdata["gex"], color=["C_scANVI"],   
           legend_fontoutline=2, 
           size=5,
           legend_fontweight='bold', 
           legend_fontsize="6", 
           color_map = 'RdYlBu_r',
           frameon=False, 
           vmin=0)

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(mdata["gex"], color=["Virus"],   
           legend_fontoutline=2, 
           legend_fontweight='bold', 
           legend_fontsize="6", 
           color_map = 'RdYlBu_r',
           size=15,
           frameon=False, 
           vmin=0)

In [ ]:
mdata["gex"].obs.Compartment = pd.Categorical(np.where(mdata["gex"].obs.Compartment.isna(), "PB", mdata["gex"].obs.Compartment))

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(mdata["gex"], color=["Compartment"],   
           legend_fontoutline=2, 
           legend_fontweight='bold', 
           legend_fontsize="6", 
           color_map = 'RdYlBu_r',
           frameon=False, 
           vmin=0)

In [ ]:
## Complete L3_ann_overview
temp_df = mdata["gex"].obs[["L3_ann", "L3_ann_overview"]].value_counts().reset_index()[["L3_ann", "L3_ann_overview"]].drop_duplicates()
state_dic = temp_df.set_index('L3_ann').to_dict()

## map the dictionary unto the sc-seq object
mdata["gex"].obs["L3_ann_overview_scanvi"] = mdata["gex"].obs['C_scANVI'].map(state_dic["L3_ann_overview"])


In [ ]:
## Adjust L3_ann_overview_scanvi
### Kilian deems the NK-like phenotype as part of the  TCM phenotype
mdata["gex"].obs["L3_ann_overview_scanvi"] = np.where(mdata["gex"].obs["L3_ann_overview_scanvi"] == "NK-like", "TCM", mdata["gex"].obs["L3_ann_overview_scanvi"])

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(mdata["gex"], color=["L3_ann_overview_scanvi"],   
           legend_fontoutline=2, 
           legend_fontweight='bold', 
           legend_fontsize="6", 
           color_map = 'RdYlBu_r',
           frameon=False, 
           vmin=0)

# Export annotations

In [ ]:
 mdata["gex"].obs.columns

In [ ]:
obs_df = mdata["gex"].obs.copy()

obs_df[obs_df["batch"] == "unannotated"][["C_scANVI", "L3_ann_overview_scanvi"]].to_csv("../../03_annotation/11_multimer_experiment_ann/1_LabelTransferObs_from_scANVI.csv")

In [ ]:
obs_df = mdata["gex"].obs.copy()

obs_df.dropna(subset="Virus")[["clone_handle", "Virus", 'handle', 'Peptide', 'Sequence', 'Virus', 'Antigen']].drop_duplicates().to_csv("../../03_annotation/11_multimer_experiment_ann/2_MultPostClones.csv")


# Plot
## Heatmap per virus-specificity
The advantage of % per virus specificity is, that we can the distribution per virus across all leiden clusters, which is less biased by the underlying cell number of each specificity

The problem with taking a heatmap per leiden cluster is, that it is biased by our capacity to detect EBV-specific cells much more efficient than other antigen-specific cells therefore we sorted sorted more of these and 1 leiden cluster could therefore be made up exclusively of one specificity, because we sorted so many EBV-specific cells. 

In [ ]:
obs_df = mdata["gex"].obs.copy()
# obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]

# Focus on PB
obs_df["L3_ann"] = obs_df["C_scANVI"].cat.remove_unused_categories()
obs_df["Virus"] = obs_df["Virus"].cat.remove_unused_categories()

# Get extra information - how many different clones per virus-specificity?
clone_count = obs_df[["clone_handle", "Virus"]].drop_duplicates().groupby("Virus", observed=True).size().reset_index(name="clone_count")
donor_count = obs_df[["donor", "Virus"]].drop_duplicates().groupby("Virus", observed=True).size().reset_index(name="donor_count")
cell_count = obs_df.groupby("Virus", observed=True).size().reset_index(name="cell_count")

# Get grouped counts per differentiation assignment
sum_df = obs_df.groupby(["Virus", "C_scANVI"], observed=False).size().reset_index(name="count")

# Add sums per leiden cluster
sum_df["total"] = sum_df.groupby(["Virus"],observed=True)["count"].transform("sum")

# Calculate percentages
sum_df["freq"] = 100 * sum_df["count"] / sum_df["total"]

# left_join
sum_df = sum_df.merge(clone_count, how="left")
sum_df = sum_df.merge(donor_count, how="left")
sum_df = sum_df.merge(cell_count, how="left")

#Modify the name
sum_df["long_name_virus"] = (
    sum_df["Virus"].astype(str)
    + " ("
    + sum_df["cell_count"].astype(str) + " cells, "
    + sum_df["clone_count"].astype(str) + " clones, "
    + sum_df["donor_count"].astype(str) + " donors)"
)

# Confusion matrix
conf_mat = (
    sum_df.pivot_table(
        index="long_name_virus",
        columns="C_scANVI",
        values="freq",
        # aggfunc="sum", # We don't need an aggregate function if each index/column  combination only has 1 entry
        fill_value=0
    )
)

# Create figure explicitly
fig, ax = plt.subplots(figsize=(17, 7))

# Heatmap
sb.heatmap(
    conf_mat,
    annot=True,
    annot_kws={"fontsize": 18},
    cmap="Reds",
    fmt=".1f",
    ax=ax
)

sb.despine()
ax.grid(False)

ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=18)
ax.set_xlabel("C_scANVI annotation", fontsize=18)
ax.set_ylabel("Virus", fontsize=18)
ax.set_title("% of multimer-specific T cells in PB")

fig.tight_layout()

# ---- SAVE FIGURE ----
fig.savefig(
    "../../05_plots/09_multimers/16_heatmap_multimer_specific_Tcells_PB_AfterIntergration.pdf",
    dpi=400,
    bbox_inches="tight"
)

plt.show()

# Bargraphs per L3_ann_overview

In [ ]:
colors = {
    "GDT cells" : "#8C6D31", 
    "NK-like" : "#279E68",
    "TA" : "#D24FA5",
    "TN" : "#D7BCA6", 
    "TCM": "#9ECAE1", 
    "TEM" : "#3182BD", 
    "TEMRA" : "#4C3696", 
    "TE" : "#C00000"
}

In [ ]:
obs_df = mdata["gex"].obs.copy()
# obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]

## dropna
obs_df = obs_df.dropna(subset="Virus")

# Remove unused categories
for col in obs_df.select_dtypes(include="category").columns:
    obs_df[col] = (
        obs_df[col].cat.remove_unused_categories()
    )

#Select columns
obs_df = obs_df[["clone_handle", "Virus", "L3_ann_overview_scanvi"]]

#Count
count_df = obs_df.groupby(["Virus", "L3_ann_overview_scanvi"]).size().reset_index(name="count")

#sum 
count_df["total_cells"] = count_df.groupby("Virus")["count"].transform("sum")

#percentage
count_df["percent_cells"] = 100*count_df["count"] / count_df["total_cells"]

# Pivot to wide format
plot_df = (
    count_df.pivot(
        index="Virus",
        columns="L3_ann_overview_scanvi",
        values="percent_cells"
    )
    .fillna(0)
)

# Optional: specify virus order
virus_order = ["CMV", "NWV", "HMPV", "FLU-A", "HHV-1", "SARS-COV-2", "HAdV-C", "EBV", "HHV-2", "VZV"]
plot_df = plot_df.reindex([v for v in virus_order if v in plot_df.index])

#Stacked bargraphs
plot_df = plot_df[[c for c in cats if c in plot_df.columns]]

fig, ax = plt.subplots(figsize=(6, 3))

plot_df.plot(
    kind="bar",
    stacked=True,
    color=[colors[c] for c in plot_df.columns],
    width=0.8,
    edgecolor="none",
    ax=ax,
)

# Axis formatting
ax.set_xlabel("")
ax.set_ylabel("% of cells")
ax.set_ylim(0, 100)
sb.despine()
ax.tick_params(axis="x", rotation=90)
ax.grid(False)

# Legend
ax.legend(
    title="Cell state",
    title_fontsize=10,
    fontsize=9,
    frameon=False,
    ncol=1,                 # Increase if many cell states
    loc="upper left",
    bbox_to_anchor=(1.02, 1),
    borderaxespad=0,
    handlelength=1.2,
    handleheight=0.8,
    labelspacing=0.4,
    columnspacing=1.2,
)

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/39_Preferred_pheno_L3overview.pdf", format="pdf", dpi=400)
plt.show()

# Statistics - TEM

In [ ]:
mdata["gex"].obs["Virus"].value_counts()

In [ ]:
obs_df = mdata["gex"].obs.copy()

# Focus on multimer annotated cells
obs_df = obs_df[~obs_df["Virus"].isna()].copy()
# obs_df = obs_df[~obs_df["catpat"].isin(["IC"])]

# Remove unused categories
for col in obs_df.select_dtypes(include="category").columns:
    obs_df[col] = (
        obs_df[col].cat.remove_unused_categories()
    )

# Categories 
template_df = obs_df.groupby(["clone_handle", "L3_ann_overview_scanvi"]).size().reset_index(name="n_cells")

# Add virus information
vir_ann = obs_df[["clone_handle", "Virus"]].drop_duplicates()
plot_df = template_df.merge(vir_ann, how="left")

# Total cells per clone_handle and Virus
plot_df["total_cells"] = (
    plot_df
    .groupby(["clone_handle", "Virus"], observed=True)["n_cells"]
    .transform("sum")
)

# Percentage of cells in each cluster
plot_df["percent_cells"] = 100 * plot_df["n_cells"] / plot_df["total_cells"]

# Focus on TEM
plot_df = plot_df[plot_df["L3_ann_overview_scanvi"] ==  "TEM"]

# Optional: specify the order of viruses
virus_order = ["CMV", "NWV", "HMPV", "HHV-6B", "FLU-A", "HHV-1", "SARS-COV-2", "HAdV-C", "EBV", "HHV-2", "VZV"]

# Keep only viruses that are present
virus_order = [v for v in virus_order if v in plot_df["Virus"].unique()]

fig, ax = plt.subplots(figsize=(4, 4))

# Boxplots
sb.boxplot(
    data=plot_df,
    x="Virus",
    y="percent_cells",
    order=virus_order,
    fliersize=0,
    width=0.5,
    color="white",
    linecolor="#4D4D4D", 
    linewidth=0.5,
    zorder=2,
    ax=ax,
)

# Set transparency of the boxes
for patch in ax.patches:
    patch.set_alpha(0.90)
    
# Individual clones
sb.stripplot(
    data=plot_df,
    x="Virus",
    y="percent_cells",
    order=virus_order,
    color="black",
    jitter=0.25,
    size=5,
    zorder=1,
    ax=ax,
)

ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="right")
ax.set_ylabel("% of cells in TEM")
ax.set_ylim(-10, 110)

sb.despine()
ax.grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/7_percentage_in_L3_overview.pdf", bbox_inches='tight')
plt.show()

In [ ]:
import scipy.stats as stats
import scikit_posthocs as sp

groups = [
    group["percent_cells"].values
    for _, group in plot_df.groupby("Virus", observed=True)
]

H, p = stats.kruskal(*groups)

print(f"Kruskal-Wallis H = {H:.3f}, p = {p:.3e}")

In [ ]:
dunn = sp.posthoc_dunn(
    plot_df,
    val_col="percent_cells",
    group_col="Virus",
    p_adjust="fdr_bh",   # or "holm", "bonferroni"
)

dunn = dunn.loc[virus_order, virus_order]

dunn_long = (
    dunn.stack()
    .reset_index()
    .rename(columns={
        "level_0": "Virus_1",
        "level_1": "Virus_2",
        0: "adjusted_p"
    })
)

# Remove self-comparisons
dunn_long = dunn_long[dunn_long["Virus_1"] != dunn_long["Virus_2"]]

# Remove duplicate comparisons
dunn_long["comparison"] = dunn_long.apply(
    lambda x: tuple(sorted((x["Virus_1"], x["Virus_2"]))),
    axis=1,
)

dunn_long = (
    dunn_long
    .drop_duplicates("comparison")
    .drop(columns="comparison")
    .sort_values("adjusted_p")
)

dunn_long.head(10)

# CSF-biased score per clone

In [ ]:
## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "CSF-biased score", "clone_handle"]].copy()
obs_df = obs_df.dropna(subset="Virus")
obs_df = obs_df.groupby(["clone_handle", "Virus"], observed=True).mean("CSF-biased score").reset_index()

#plotting order: 
order = (
    obs_df
    .groupby("Virus")["CSF-biased score"]
    .median()
    .sort_values()
    .index
)

# Create the figure object
fig, ax = plt.subplots(figsize=(4,4))

# Stripplot (background dots)
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="CSF-biased score",
    color="black",
    jitter=0.25,
    size=5,
    zorder=1,
    order=order,
    ax=ax
)

# Boxplot (foreground)
sb.boxplot(
    data=obs_df,
    x="Virus",
    y="CSF-biased score",
    order=order,
    fliersize=0,
    width=0.5,
    color="white",
    linecolor="#4D4D4D", 
    linewidth=0.5,
    zorder=2,
    ax=ax
)

plt.ylabel("Mean CSF-biased score\nper clone")
plt.xlabel("Virus")
plt.xticks(rotation=90)
sb.despine()

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/7_mean_clone_CSF_biased_score.pdf", bbox_inches='tight')
plt.show()

In [ ]:
import scipy.stats as stats
import scikit_posthocs as sp

## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "CSF-biased score", "clone_handle"]].copy()
obs_df = obs_df.dropna(subset="Virus")
obs_df = obs_df.groupby(["clone_handle", "Virus"], observed=True).mean("PB-biased score").reset_index()

groups = [
    group["CSF-biased score"].values
    for _, group in obs_df.groupby("Virus", observed=True)
]

H, p = stats.kruskal(*groups)

print(f"Kruskal-Wallis H = {H:.3f}, p = {p:.3e}")

In [ ]:
dunn = sp.posthoc_dunn(
    obs_df,
    val_col="CSF-biased score",
    group_col="Virus",
    p_adjust="fdr_bh",   # or "holm", "bonferroni"
)

dunn = dunn.loc[virus_order, virus_order]

dunn_long = (
    dunn.stack()
    .reset_index()
    .rename(columns={
        "level_0": "Virus_1",
        "level_1": "Virus_2",
        0: "adjusted_p"
    })
)

# Remove self-comparisons
dunn_long = dunn_long[dunn_long["Virus_1"] != dunn_long["Virus_2"]]

# Remove duplicate comparisons
dunn_long["comparison"] = dunn_long.apply(
    lambda x: tuple(sorted((x["Virus_1"], x["Virus_2"]))),
    axis=1,
)

dunn_long = (
    dunn_long
    .drop_duplicates("comparison")
    .drop(columns="comparison")
    .sort_values("adjusted_p")
)

dunn_long.head(10)

# PB-biased score per clone

In [ ]:
## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "PB-biased score", "clone_handle"]].copy()
obs_df = obs_df.dropna(subset="Virus")
obs_df = obs_df.groupby(["clone_handle", "Virus"], observed=True).mean("PB-biased score").reset_index()

# Create the figure object
fig, ax = plt.subplots(figsize=(4,4))

# Stripplot (background dots)
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="PB-biased score",
    color="black",
    jitter=0.25,
    size=5,
    zorder=1,
    order=order,
    ax=ax
)

# Boxplot (foreground)
sb.boxplot(
    data=obs_df,
    x="Virus",
    y="PB-biased score",
    order=order,
    fliersize=0,
    width=0.5,
    color="white",
    linecolor="#4D4D4D", 
    linewidth=0.5,
    zorder=2,
    ax=ax
)

plt.ylabel("Mean PB-biased score\nper clone")
plt.xlabel("Virus")
plt.xticks(rotation=90)
sb.despine()

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/7_mean_clone_PB_biased_score.pdf", bbox_inches='tight')
plt.show()

In [ ]:
import scipy.stats as stats
import scikit_posthocs as sp

## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "PB-biased score", "clone_handle"]].copy()
obs_df = obs_df.dropna(subset="Virus")
obs_df = obs_df.groupby(["clone_handle", "Virus"], observed=True).mean("PB-biased score").reset_index()

groups = [
    group["PB-biased score"].values
    for _, group in obs_df.groupby("Virus", observed=True)
]

H, p = stats.kruskal(*groups)

print(f"Kruskal-Wallis H = {H:.3f}, p = {p:.3e}")

In [ ]:
dunn = sp.posthoc_dunn(
    obs_df,
    val_col="PB-biased score",
    group_col="Virus",
    p_adjust="fdr_bh",   # or "holm", "bonferroni"
)

dunn = dunn.loc[virus_order, virus_order]

dunn_long = (
    dunn.stack()
    .reset_index()
    .rename(columns={
        "level_0": "Virus_1",
        "level_1": "Virus_2",
        0: "adjusted_p"
    })
)

# Remove self-comparisons
dunn_long = dunn_long[dunn_long["Virus_1"] != dunn_long["Virus_2"]]

# Remove duplicate comparisons
dunn_long["comparison"] = dunn_long.apply(
    lambda x: tuple(sorted((x["Virus_1"], x["Virus_2"]))),
    axis=1,
)

dunn_long = (
    dunn_long
    .drop_duplicates("comparison")
    .drop(columns="comparison")
    .sort_values("adjusted_p")
)

dunn_long.head(10)

# Statistics - TE

In [ ]:
mdata["gex"].obs["Virus"].value_counts()

In [ ]:
obs_df = mdata["gex"].obs.copy()

# Focus on multimer annotated cells
obs_df = obs_df[~obs_df["Virus"].isna()].copy()

# Remove unused categories
for col in obs_df.select_dtypes(include="category").columns:
    obs_df[col] = (
        obs_df[col].cat.remove_unused_categories()
    )

# Categories 
template_df = obs_df.groupby(["clone_handle", "L3_ann_overview_scanvi"]).size().reset_index(name="n_cells")

# Add virus information
vir_ann = obs_df[["clone_handle", "Virus"]].drop_duplicates()
plot_df = template_df.merge(vir_ann, how="left")

# Total cells per clone_handle and Virus
plot_df["total_cells"] = (
    plot_df
    .groupby(["clone_handle", "Virus"], observed=True)["n_cells"]
    .transform("sum")
)

# Percentage of cells in each cluster
plot_df["percent_cells"] = 100 * plot_df["n_cells"] / plot_df["total_cells"]

# Focus on TEM
plot_df = plot_df[plot_df["L3_ann_overview_scanvi"] ==  "TE"]

# Optional: specify the order of viruses
virus_order = ["CMV", "NWV", "HMPV", "HHV-6B", "FLU-A", "HHV-1", "SARS-COV-2", "HAdV-C", "EBV", "HHV-2", "VZV"]

# Keep only viruses that are present
virus_order = [v for v in virus_order if v in plot_df["Virus"].unique()]

fig, ax = plt.subplots(figsize=(4, 4))

# Boxplots
sb.boxplot(
    data=plot_df,
    x="Virus",
    y="percent_cells",
    order=virus_order,
    fliersize=0,
    width=0.5,
    color="white",
    linecolor="#4D4D4D", 
    linewidth=0.5,
    zorder=2,
    ax=ax,
)

# Set transparency of the boxes
for patch in ax.patches:
    patch.set_alpha(0.90)
    
# Individual clones
sb.stripplot(
    data=plot_df,
    x="Virus",
    y="percent_cells",
    order=virus_order,
    color="black",
    jitter=0.25,
    size=5,
    zorder=1,
    ax=ax,
)

# # Violin plots
# sb.violinplot(
#     data=plot_df,
#     x="Virus",
#     y="percent_cells",
#     order=virus_order,
#     inner=None,
#     cut=0,
#     color="white",
#     linewidth=0.5,
#     linecolor="#4D4D4D",
#     width=0.8,
#     density_norm="width",   # all violins have the same maximum width
#     ax=ax,
# )

# # Make violins slightly transparent
# for violin in ax.collections:
#     violin.set_alpha(0.9)

# # Individual clones
# sb.stripplot(
#     data=plot_df,
#     x="Virus",
#     y="percent_cells",
#     order=virus_order,
#     color="black",
#     jitter=0.1,
#     alpha=0.5,
#     size=5,
#     zorder=3,
#     ax=ax,
# )

ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="right")
ax.set_ylabel("% of cells in TE")
ax.set_ylim(-10, 110)

sb.despine()
ax.grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/7_percentage_TE_in_L3_overview.pdf", bbox_inches='tight')
plt.show()

In [ ]:
# import scipy.stats as stats
import scikit_posthocs as sp

groups = [
    group["percent_cells"].values
    for _, group in plot_df.groupby("Virus", observed=True)
]

H, p = stats.kruskal(*groups)

print(f"Kruskal-Wallis H = {H:.3f}, p = {p:.3e}")

In [ ]:
dunn = sp.posthoc_dunn(
    plot_df,
    val_col="percent_cells",
    group_col="Virus",
    p_adjust="fdr_bh",   # or "holm", "bonferroni"
)

dunn = dunn.loc[virus_order, virus_order]

dunn_long = (
    dunn.stack()
    .reset_index()
    .rename(columns={
        "level_0": "Virus_1",
        "level_1": "Virus_2",
        0: "adjusted_p"
    })
)

# Remove self-comparisons
dunn_long = dunn_long[dunn_long["Virus_1"] != dunn_long["Virus_2"]]

# Remove duplicate comparisons
dunn_long["comparison"] = dunn_long.apply(
    lambda x: tuple(sorted((x["Virus_1"], x["Virus_2"]))),
    axis=1,
)

dunn_long = (
    dunn_long
    .drop_duplicates("comparison")
    .drop(columns="comparison")
    .sort_values("adjusted_p")
)

dunn_long.head(10)

# Statistics - TCM

In [ ]:
mdata["gex"].obs["Virus"].value_counts()

In [ ]:
obs_df = mdata["gex"].obs.copy()

# Focus on multimer annotated cells
obs_df = obs_df[~obs_df["Virus"].isna()].copy()

# Remove unused categories
for col in obs_df.select_dtypes(include="category").columns:
    obs_df[col] = (
        obs_df[col].cat.remove_unused_categories()
    )

# Categories 
template_df = obs_df.groupby(["clone_handle", "L3_ann_overview_scanvi"]).size().reset_index(name="n_cells")

# Add virus information
vir_ann = obs_df[["clone_handle", "Virus"]].drop_duplicates()
plot_df = template_df.merge(vir_ann, how="left")

# Total cells per clone_handle and Virus
plot_df["total_cells"] = (
    plot_df
    .groupby(["clone_handle", "Virus"], observed=True)["n_cells"]
    .transform("sum")
)

# Percentage of cells in each cluster
plot_df["percent_cells"] = 100 * plot_df["n_cells"] / plot_df["total_cells"]

# Focus on TEM
plot_df = plot_df[plot_df["L3_ann_overview_scanvi"] ==  "TCM"]

# Optional: specify the order of viruses
virus_order = ["CMV", "NWV", "HMPV", "HHV-6B", "FLU-A", "HHV-1", "SARS-COV-2", "HAdV-C", "EBV", "HHV-2", "VZV"]

# Keep only viruses that are present
virus_order = [v for v in virus_order if v in plot_df["Virus"].unique()]

fig, ax = plt.subplots(figsize=(4, 4))

# Boxplots
sb.boxplot(
    data=plot_df,
    x="Virus",
    y="percent_cells",
    order=virus_order,
    fliersize=0,
    width=0.5,
    color="white",
    linecolor="#4D4D4D", 
    linewidth=0.5,
    zorder=2,
    ax=ax,
)

# Set transparency of the boxes
for patch in ax.patches:
    patch.set_alpha(0.90)
    
# Individual clones
sb.stripplot(
    data=plot_df,
    x="Virus",
    y="percent_cells",
    order=virus_order,
    color="black",
    jitter=0.25,
    size=5,
    zorder=1,
    ax=ax,
)

# # Violin plots
# sb.violinplot(
#     data=plot_df,
#     x="Virus",
#     y="percent_cells",
#     order=virus_order,
#     inner=None,
#     cut=0,
#     color="white",
#     linewidth=0.5,
#     linecolor="#4D4D4D",
#     width=0.8,
#     density_norm="width",   # all violins have the same maximum width
#     ax=ax,
# )

# # Make violins slightly transparent
# for violin in ax.collections:
#     violin.set_alpha(0.9)

# # Individual clones
# sb.stripplot(
#     data=plot_df,
#     x="Virus",
#     y="percent_cells",
#     order=virus_order,
#     color="black",
#     jitter=0.1,
#     alpha=0.5,
#     size=5,
#     zorder=3,
#     ax=ax,
# )

ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="right")
ax.set_ylabel("% of cells in TCM")
ax.set_ylim(-10, 110)

sb.despine()
ax.grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/7_percentage_TCM_in_L3_overview.pdf", bbox_inches='tight')
plt.show()

In [ ]:
import scipy.stats as stats
import scikit_posthocs as sp

groups = [
    group["percent_cells"].values
    for _, group in plot_df.groupby("Virus", observed=True)
]

H, p = stats.kruskal(*groups)

print(f"Kruskal-Wallis H = {H:.3f}, p = {p:.3e}")

In [ ]:
dunn = sp.posthoc_dunn(
    plot_df,
    val_col="percent_cells",
    group_col="Virus",
    p_adjust="fdr_bh",   # or "holm", "bonferroni"
)

dunn = dunn.loc[virus_order, virus_order]

dunn_long = (
    dunn.stack()
    .reset_index()
    .rename(columns={
        "level_0": "Virus_1",
        "level_1": "Virus_2",
        0: "adjusted_p"
    })
)

# Remove self-comparisons
dunn_long = dunn_long[dunn_long["Virus_1"] != dunn_long["Virus_2"]]

# Remove duplicate comparisons
dunn_long["comparison"] = dunn_long.apply(
    lambda x: tuple(sorted((x["Virus_1"], x["Virus_2"]))),
    axis=1,
)

dunn_long = (
    dunn_long
    .drop_duplicates("comparison")
    .drop(columns="comparison")
    .sort_values("adjusted_p")
)

dunn_long.head(10)

# Statistics - MS vs NIC

In [ ]:
obs_df = mdata["gex"].obs[["Virus", "CSF-biased score", "clone_handle", "catpat"]].copy()
obs_df["catpat"].value_counts()

In [ ]:
## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "CSF-biased score", "clone_handle", "catpat"]].copy()
obs_df = obs_df[obs_df["catpat"].isin(["NIC", "MS"])]
obs_df = obs_df.dropna(subset="Virus")
obs_df = obs_df.groupby(["clone_handle", "Virus", "catpat"], observed=True).mean("CSF-biased score").reset_index()
obs_df = obs_df[obs_df["Virus"].isin(["CMV", "FLU-A", "EBV", "SARS-COV-2"])]
print(obs_df["catpat"].value_counts())
# Remove unused categories
for col in obs_df.select_dtypes(include="category").columns:
    obs_df[col] = (
        obs_df[col].cat.remove_unused_categories()
    )

#Define order through categorical levels
obs_df["catpat"] = pd.Categorical(obs_df["catpat"], categories=["NIC", "MS"])
    
# Define manual colors
palette = {
    "NIC": "#FFFFFF",   # blue
    "MS": "#000000"       # red
}

#Defin order:
order_vir = ["CMV", "FLU-A", "SARS-COV-2", "EBV"]

# Create the figure object
fig, ax = plt.subplots(figsize=(4,4))

# Stripplot (background dots)
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="CSF-biased score",
    hue="catpat",
    edgecolor="black",
    linewidth=0.5,
    palette=palette,
    dodge=True,
    jitter=0.25,
    size=5,
    alpha=0.8,
    zorder=1,
    order=order_vir,
    ax=ax
)

# Boxplot (foreground)
sb.boxplot(
    data=obs_df,
    x="Virus",
    y="CSF-biased score",
    hue="catpat",
    palette=["white", "white"],   # white boxes for both groups
    dodge=True,
    order=order_vir,
    fliersize=0,
    linecolor="#4D4D4D",
    linewidth=0.5,
    zorder=2,
    ax=ax
)

# Remove duplicate legends and create one clean legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[:2],
    labels[:2],
    title="Group",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
)

sb.despine()
ax.grid(False)


plt.ylabel("Mean CSF-biased score")
plt.xlabel("Virus")
plt.xticks(rotation=90)

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/7_mean_clone_CSF_NICvMS_CSFbiased_score.pdf", bbox_inches='tight')
plt.show()

In [ ]:
results = []

for virus, sub in obs_df.groupby("Virus", observed=True):

    control = sub.loc[sub["catpat"] == "NIC", "CSF-biased score"]
    rrms = sub.loc[sub["catpat"] == "MS", "CSF-biased score"]

    # Skip if either group is empty
    if len(control) == 0 or len(rrms) == 0:
        continue

    stat, p = mannwhitneyu(
        control,
        rrms,
        alternative="two-sided",
        method="exact"
    )

    results.append({
        "Virus": virus,
        "n_Control": len(control),
        "n_RRMS": len(rrms),
        "U": stat,
        "p": p,
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("Virus")
    .reset_index(drop=True)
)

results_df

# CSF-biased score per donor

In [ ]:
## Extract score values per cell and plot as a box plot per specificity
obs_df = mdata["gex"].obs[["Virus", "CSF-biased score", "donor", "catpat", "total_clone_count"]].copy()
obs_df = obs_df[obs_df["catpat"].isin(["NIC", "MS"])]
obs_df = obs_df.dropna(subset="Virus")
obs_df = obs_df.groupby(["donor", "Virus", "catpat"], observed=True).mean("CSF-biased score").reset_index()
obs_df = obs_df[obs_df["Virus"].isin(["CMV", "FLU-A", "EBV", "SARS-COV-2"])]

# Remove unused categories
for col in obs_df.select_dtypes(include="category").columns:
    obs_df[col] = (
        obs_df[col].cat.remove_unused_categories()
    )

#Define order through categorical levels
obs_df["catpat"] = pd.Categorical(obs_df["catpat"], categories=["NIC", "MS"])
    
# Define manual colors
palette = {
    "NIC": "#FFFFFF",   # blue
    "MS": "#000000"       # red
}

#Defin order:
order_vir = ["CMV", "FLU-A", "SARS-COV-2", "EBV"]

# Create the figure object
fig, ax = plt.subplots(figsize=(4,4))

# Stripplot (background dots)
sb.stripplot(
    data=obs_df,
    x="Virus",
    y="CSF-biased score",
    hue="catpat",
    edgecolor="black",
    linewidth=0.5,
    palette=palette,
    dodge=True,
    jitter=0.25,
    size=5,
    alpha=0.8,
    zorder=1,
    order=order_vir,
    ax=ax
)

# Boxplot (foreground)
sb.boxplot(
    data=obs_df,
    x="Virus",
    y="CSF-biased score",
    hue="catpat",
    palette=["white", "white"],   # white boxes for both groups
    dodge=True,
    order=order_vir,
    fliersize=0,
    linecolor="#4D4D4D",
    linewidth=0.5,
    zorder=2,
    ax=ax
)

# Set transparency of the boxes
for patch in ax.patches:
    patch.set_alpha(0.60)


# Remove duplicate legends and create one clean legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[:2],
    labels[:2],
    title="Group",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
)

sb.despine()
ax.grid(False)
ax.set_ylim(-0.25, 0.45)


plt.ylabel("Mean CSF-biased score\nper donor")
plt.xlabel("Virus")
plt.xticks(rotation=90)

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/7_mean_clone_CSF_NICvMS_CSFbiased_score_perdonor.pdf", bbox_inches='tight')
plt.show()

# % TEM

In [ ]:
obs_df = mdata["gex"].obs.copy()

# Focus on multimer annotated cells
obs_df = obs_df[~obs_df["Virus"].isna()].copy()
obs_df = obs_df[obs_df["catpat"].isin(["NIC", "MS"])]

# Only work with 4 viruses with sufficient number of responses
obs_df = obs_df[obs_df["Virus"].isin(["CMV", "FLU-A", "EBV", "SARS-COV-2"])]

# Remove unused categories
for col in obs_df.select_dtypes(include="category").columns:
    obs_df[col] = (
        obs_df[col].cat.remove_unused_categories()
    )

# Categories 
template_df = obs_df.groupby(["clone_handle", "L3_ann_overview_scanvi"]).size().reset_index(name="n_cells")

# Add virus information
vir_ann = obs_df[["clone_handle", "Virus"]].drop_duplicates()
catpat_ann =  obs_df[["clone_handle", "catpat"]].drop_duplicates()
plot_df = template_df.merge(vir_ann, how="left").merge(catpat_ann, how="left")

# Total cells per clone_handle and Virus
plot_df["total_cells"] = (
    plot_df
    .groupby(["clone_handle", "Virus"], observed=True)["n_cells"]
    .transform("sum")
)

# Percentage of cells in each cluster
plot_df["percent_cells"] = 100 * plot_df["n_cells"] / plot_df["total_cells"]

# Focus on TEM
plot_df = plot_df[plot_df["L3_ann_overview_scanvi"] ==  "TEM"]

# Optional: specify the order of viruses
virus_order = ["CMV", "FLU-A", "EBV", "SARS-COV-2"]

# Keep only viruses that are present
virus_order = [v for v in virus_order if v in plot_df["Virus"].unique()]

#Define order through categorical levels
plot_df["catpat"] = pd.Categorical(plot_df["catpat"], categories=["NIC", "MS"])
    
# Define manual colors
palette = {
    "NIC": "#FFFFFF",   # blue
    "MS": "#000000"       # red
}

fig, ax = plt.subplots(figsize=(4.5, 4))

# Stripplot (background dots)
sb.stripplot(
    data=plot_df,
    x="Virus",
    y="percent_cells",
    hue="catpat",
    edgecolor="black",
    linewidth=0.5,
    palette=palette,
    dodge=True,
    jitter=0.25,
    size=5,
    alpha=0.8,
    zorder=1,
    order=order_vir,
    ax=ax
)

# Boxplot (foreground)
sb.boxplot(
    data=plot_df,
    x="Virus",
    y="percent_cells",
    hue="catpat",
    palette=["white", "white"],   # white boxes for both groups
    dodge=True,
    order=order_vir,
    fliersize=0,
    linecolor="#4D4D4D",
    linewidth=0.5,
    zorder=2,
    ax=ax
)


# Set transparency of the boxes
for patch in ax.patches:
    patch.set_alpha(0.60)

# Remove duplicate legends and create one clean legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[:2],
    labels[:2],
    title="Group",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
)

ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="right")
ax.set_ylabel("% of cells in TEM")
ax.set_ylim(-10, 110)

sb.despine()
ax.grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/7_percentage_in_L3_overview.pdf", bbox_inches='tight')
plt.show()

In [ ]:
results = []

for virus, sub in plot_df.groupby("Virus", observed=True):

    control = sub.loc[sub["catpat"] == "NIC", "percent_cells"]
    rrms = sub.loc[sub["catpat"] == "MS", "percent_cells"]

    # Skip if either group is empty
    if len(control) == 0 or len(rrms) == 0:
        continue

    stat, p = mannwhitneyu(
        control,
        rrms,
        alternative="two-sided",
        method="exact"
    )

    results.append({
        "Virus": virus,
        "n_Control": len(control),
        "n_RRMS": len(rrms),
        "U": stat,
        "p": p,
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("Virus")
    .reset_index(drop=True)
)

results_df

# % TEM per donor

In [ ]:
obs_df = mdata["gex"].obs.copy()

# Focus on multimer annotated cells
obs_df = obs_df[~obs_df["Virus"].isna()].copy()
obs_df = obs_df[obs_df["catpat"].isin(["NIC", "MS"])]

# Only work with 4 viruses with sufficient number of responses
obs_df = obs_df[obs_df["Virus"].isin(["CMV", "FLU-A", "EBV", "SARS-COV-2"])]

# Remove unused categories
for col in obs_df.select_dtypes(include="category").columns:
    obs_df[col] = (
        obs_df[col].cat.remove_unused_categories()
    )

# Categories 
template_df = obs_df.groupby(["donor", "Virus", "L3_ann_overview_scanvi"]).size().reset_index(name="n_cells")

# Add virus information
vir_ann = obs_df[["donor", "Virus"]].drop_duplicates()
catpat_ann =  obs_df[["donor", "catpat"]].drop_duplicates()
plot_df = template_df.merge(vir_ann, how="left").merge(catpat_ann, how="left")

# Total cells per clone_handle and Virus
plot_df["total_cells"] = (
    plot_df
    .groupby(["Virus", "donor"], observed=True)["n_cells"]
    .transform("sum")
)

# Percentage of cells in each cluster
plot_df["percent_cells"] = 100 * plot_df["n_cells"] / plot_df["total_cells"]

# Focus on TEM
plot_df = plot_df[plot_df["L3_ann_overview_scanvi"] ==  "TEM"]

# Optional: specify the order of viruses
virus_order = ["CMV", "FLU-A", "EBV", "SARS-COV-2"]

# Keep only viruses that are present
virus_order = [v for v in virus_order if v in plot_df["Virus"].unique()]

#Define order through categorical levels
plot_df["catpat"] = pd.Categorical(plot_df["catpat"], categories=["NIC", "MS"])
    
# Define manual colors
palette = {
    "NIC": "#FFFFFF",   # blue
    "MS": "#000000"       # red
}

fig, ax = plt.subplots(figsize=(4.5, 4))

# Stripplot (background dots)
sb.stripplot(
    data=plot_df,
    x="Virus",
    y="percent_cells",
    hue="catpat",
    edgecolor="black",
    linewidth=0.5,
    palette=palette,
    dodge=True,
    jitter=0.25,
    size=5,
    alpha=0.8,
    zorder=1,
    order=order_vir,
    ax=ax
)

# Boxplot (foreground)
sb.boxplot(
    data=plot_df,
    x="Virus",
    y="percent_cells",
    hue="catpat",
    palette=["white", "white"],   # white boxes for both groups
    dodge=True,
    order=order_vir,
    fliersize=0,
    linecolor="#4D4D4D",
    linewidth=0.5,
    zorder=2,
    ax=ax
)


# Set transparency of the boxes
for patch in ax.patches:
    patch.set_alpha(0.60)

# Remove duplicate legends and create one clean legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[:2],
    labels[:2],
    title="Group",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
)

ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="right")
ax.set_ylabel("% of cells in TEM")
ax.set_ylim(-10, 110)

sb.despine()
ax.grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/7_percentage_in_L3_overview_NICvMS.pdf", bbox_inches='tight')
plt.show()

# Original - hyperexpanded clones

In [ ]:
## Virus/multimer annotation
obs_df = mdata["gex"].obs.copy()

# Focus on multimer annotated cells
obs_df = obs_df[~obs_df["Virus"].isna()].copy()
obs_df = obs_df[obs_df["catpat"].isin(["NIC", "MS"])]

# Remove unused categories
for col in obs_df.select_dtypes(include="category").columns:
    obs_df[col] = (
        obs_df[col].cat.remove_unused_categories()
    )

mult_ann = obs_df[["clone_handle", "Virus", "HLA", "Sequence"]].drop_duplicates()

# Original hyperexpanded clones
clone_table = pd.read_csv("../../06_csv_outputs/Clones/2026_7_CD8_hyperexpanded_clone_handles_and_composition_table_across_compartments.csv")

# Join
plot_df = clone_table.merge(mult_ann, how="left")

# Add no epiope
plot_df["Virus"] = np.where(plot_df["Virus"].isna(), "Not annotated", plot_df["Virus"])

# Redefine the donor column
plot_df["donor"] = plot_df["clone_handle"].str.split('-').str[0]

# count how many we deorphanized per donor
count_df = plot_df.groupby(["donor", "Virus"],observed=True).size().reset_index(name="count")

#Colors
virus_colors = {
    "EBV": "#DD8452",
    "CMV": "#4C72B0",
    "FLU-A": "#55A868",
    "VZV": "#17BECF",
    "Not annotated" : "#D3D3D3"
}


# Pivot to wide format
plot_df = (
    count_df
    .pivot(
        index="donor",
        columns="Virus",
        values="count"
    )
    .fillna(0)
)

# Reorder columns according to dictionary order
plot_df = plot_df[
    [v for v in virus_colors.keys() if v in plot_df.columns]
]

# Ensure colors follow the plotted column order
colors = [
    virus_colors[v]
    for v in plot_df.columns
]


# Apply desired donor order
ordered_donor_list = ["Pt1","Pt8", "Pt18", "Pt19", "Pt21",  "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
plot_df = plot_df.reindex(
    [d for d in ordered_donor_list if d in plot_df.index]
)

# Plot
fig, ax = plt.subplots(figsize=(6, 4))

plot_df.plot(
    kind="bar",
    stacked=True,
    color=colors,
    width=0.8,
    edgecolor="none",
    ax=ax
)

ax.set_ylabel("Number of clones")
ax.set_xlabel("Donor")

sb.despine()
ax.grid(False)

ax.legend(
    title="Virus",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/42_Deorph_hypexp.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
## Virus/multimer annotation
obs_df = mdata["gex"].obs.copy()

# Focus on multimer annotated cells
obs_df = obs_df[~obs_df["Virus"].isna()].copy()
obs_df = obs_df[obs_df["catpat"].isin(["MS", "NIC"])]

# Remove unused categories
for col in obs_df.select_dtypes(include="category").columns:
    obs_df[col] = (
        obs_df[col].cat.remove_unused_categories()
    )

mult_ann = obs_df[["clone_handle", "Virus", "HLA", "Sequence"]].drop_duplicates()

# Original hyperexpanded clones
clone_table = pd.read_csv("../../06_csv_outputs/Clones/2026_7_CD8_hyperexpanded_clone_handles_and_composition_table_across_compartments.csv")

# Join
plot_df = clone_table.merge(mult_ann, how="left")

# dropna
plot_df2 = plot_df.dropna(subset="Virus")

# order by CSF.pct and plot stacked bargraphs with CSF.pct at the bottom and PB.pct on top with the sequence column on the x-axis
# Sort by CSF contribution
plot_df2 = plot_df2.sort_values("CSF.pct", ascending=False)

# Select relevant columns
plot_long = plot_df2.melt(
    id_vars=["clone_handle", "Sequence", "Virus", "HLA"],
    value_vars=["CSF.pct", "PB.pct"],
    var_name="Compartment",
    value_name="Percent"
)
print(plot_long.head(2))

# Make sure CSF is plotted first (bottom)
plot_long["Compartment"] = pd.Categorical(
    plot_long["Compartment"],
    categories=["CSF.pct", "PB.pct"],
    ordered=True
)

# Pivot for stacked barplot
plot_pivot = (
    plot_long
    .pivot(
        index="clone_handle",
        columns="Compartment",
        values="Percent"
    )
    .fillna(0)
)

# Ensure order
plot_pivot = plot_pivot[["CSF.pct", "PB.pct"]]

# Preserve CSF.pct sorting
clone_order = plot_df2["clone_handle"].drop_duplicates()
plot_pivot = plot_pivot.reindex(clone_order)

#Colors
virus_colors = {
    "EBV": "#DD8452",
    "CMV": "#4C72B0",
    "FLU-A": "#55A868",
    "HHV-1": "#9467BD",
    "HHV-2": "#8E1EF4",
    "VZV": "#17BECF",
    "HMPV": "#E377C2",
    "SARS-COV-2": "#BCBD22",
    "HAdV-C": "#D62728",
    "NWV": "#7F7F7F"
}

# Clone order from your stacked barplot
clone_order = plot_pivot.index

# Match virus annotation to clone order
virus_track = (
    plot_df2
    .drop_duplicates("clone_handle")
    .set_index("clone_handle")
    .loc[clone_order, "Virus"]
)

# Convert virus labels to colors
virus_rgb = [
    virus_colors[v] for v in virus_track
]


fig, (ax, ax_virus) = plt.subplots(
    nrows=2,
    figsize=(12, 6),
    gridspec_kw={"height_ratios": [8, 0.4]},
    sharex=True
)

# --------------------
# Main stacked barplot
# --------------------
plot_pivot.plot(
    kind="bar",
    stacked=True,
    color=["#4B8AC1", "#C06A6A"],
    width=1,
    edgecolor="none",
    ax=ax
)

ax.set_ylabel("Compartment (%)")
ax.set_xlabel("")
ax.set_ylim(0, 100)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    ["CSF", "PB"],
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.02, 1)
)


# --------------------
# Virus annotation bar
# --------------------
ax_virus.imshow(
    np.array([[
        plt.matplotlib.colors.to_rgb(c)
        for c in virus_rgb
    ]]),
    aspect="auto"
)

ax_virus.set_yticks([])
ax_virus.set_xticks([])

for spine in ax_virus.spines.values():
    spine.set_visible(False)


# Virus legend
virus_legend = [
    Patch(facecolor=color, label=virus)
    for virus, color in virus_colors.items()
    if virus in virus_track.values
]

ax.legend(
    handles=[
        Patch(facecolor="#4B8AC1", label="CSF"),
        Patch(facecolor="#C06A6A", label="PB"),
    ],
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.02, 1)
)

ax_virus.legend(
    handles=virus_legend,
    title="Virus specificity",
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

ax.grid(False)

plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()

In [ ]:
# Keep catpat information
mult_ann = obs_df[["clone_handle", "Virus", "HLA", "Sequence", "catpat"]].drop_duplicates()

# Join
plot_df = clone_table.merge(
    mult_ann,
    how="left"
)

# Drop unannotated
plot_df2 = plot_df.dropna(subset="Virus").copy()

# Sort by CSF contribution
plot_df2 = plot_df2.sort_values("CSF.pct", ascending=False)

# Select relevant columns
plot_long = plot_df2.melt(
    id_vars=["clone_handle", "Sequence", "Virus", "HLA", "catpat"],
    value_vars=["CSF.pct", "PB.pct"],
    var_name="Compartment",
    value_name="Percent"
)

# Pivot for stacked barplot
plot_pivot = (
    plot_long
    .pivot(
        index="clone_handle",
        columns="Compartment",
        values="Percent"
    )
    .fillna(0)
)

plot_pivot = plot_pivot[["CSF.pct", "PB.pct"]]

# Preserve CSF sorting
clone_order = plot_df2["clone_handle"].drop_duplicates()
plot_pivot = plot_pivot.reindex(clone_order)


# Colors
virus_colors = {
    "EBV": "#DD8452",
    "CMV": "#4C72B0",
    "FLU-A": "#55A868",
    "HHV-1": "#9467BD",
    "HHV-2": "#8E1EF4",
    "VZV": "#17BECF",
    "HMPV": "#E377C2",
    "SARS-COV-2": "#BCBD22",
    "HAdV-C": "#D62728",
    "NWV": "#7F7F7F"
}

catpat_colors = {
    "NIC": "#FFFFFF",
    "MS": "#000000"
}


# Match annotations to clone order
annotation_df = (
    plot_df2
    .drop_duplicates("clone_handle")
    .set_index("clone_handle")
    .loc[clone_order]
)

virus_track = annotation_df["Virus"]
catpat_track = annotation_df["catpat"]

virus_rgb = [
    plt.matplotlib.colors.to_rgb(virus_colors[v])
    for v in virus_track
]

catpat_rgb = [
    plt.matplotlib.colors.to_rgb(catpat_colors[v])
    for v in catpat_track
]


# Figure with two annotation tracks
fig, (ax, ax_virus, ax_catpat) = plt.subplots(
    nrows=3,
    figsize=(6, 6),
    gridspec_kw={
        "height_ratios": [8, 0.4, 0.4]
    },
    sharex=True
)


# --------------------
# Main stacked barplot
# --------------------
plot_pivot.plot(
    kind="bar",
    stacked=True,
    color=["#4B8AC1", "#C06A6A"],
    width=1,
    edgecolor="none",
    ax=ax
)

ax.set_ylabel("Compartment (%)")
ax.set_xlabel("")
ax.set_ylim(0, 100)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    handles=[
        Patch(facecolor="#4B8AC1", label="CSF"),
        Patch(facecolor="#C06A6A", label="PB"),
    ],
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.02, 1)
)

ax.grid(False)


# --------------------
# Virus annotation strip
# --------------------
ax_virus.imshow(
    np.array([virus_rgb]),
    aspect="auto",
    interpolation="nearest"
)

ax_virus.set_yticks([])
ax_virus.set_xticks([])


# --------------------
# Cohort annotation strip
# --------------------
ax_catpat.imshow(
    np.array([catpat_rgb]),
    aspect="auto",
    interpolation="nearest"
)

ax_catpat.set_yticks([])
ax_catpat.set_xticks([])


# Remove annotation axes borders
for annotation_ax in [ax_virus, ax_catpat]:
    for spine in annotation_ax.spines.values():
        spine.set_visible(False)


# Legends
virus_legend = [
    Patch(facecolor=color, label=virus)
    for virus, color in virus_colors.items()
    if virus in virus_track.values
]

ax_virus.legend(
    handles=virus_legend,
    title="Virus specificity",
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

catpat_legend = [
    Patch(facecolor=color, label=group)
    for group, color in catpat_colors.items()
    if group in catpat_track.values
]

ax_catpat.legend(
    handles=catpat_legend,
    title="Cohort",
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)


plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/40_hyperexpanded_clone_deorphanization.pdf")